In [15]:
import databento as db
import inspect
import pandas as pd
from pathlib import Path

In [10]:
data_folder = Path("../data/market_data/raw/FDAX/")

In [16]:
store = db.DBNStore.from_file(data_folder / "xeur-eobi-20250502.mbo.dbn.zst")
df = next(store.to_df(count=1000)).reset_index()

# Convert Timestamp columns to raw int64 nanoseconds for arithmetic
ts_event_ns = df["ts_event"].astype("int64")
ts_recv_ns  = df["ts_recv"].astype("int64")

# Check timestamp precision
print("=== Timestamp precision (raw ns) ===")
print(ts_event_ns.head(20).tolist())

print("\nLast 3 digits of ts_event (000 = microsecond precision):")
print((ts_event_ns % 1000).value_counts())

print("\nLast 3 digits of ts_recv (000 = microsecond precision):")
print((ts_recv_ns % 1000).value_counts())

# Channel IDs
print("\n=== Channel IDs ===")
print(df["channel_id"].value_counts())

# Flags
print("\n=== Flags ===")
print(df["flags"].value_counts())

# Price scale
print("\n=== Raw prices (first 5 trades) ===")
trades = df[df["action"] == "T"][["ts_event", "price", "size"]].head(5)
print(trades)
if not trades.empty:
    print("price / 1e9 =", (trades["price"] / 1e9).tolist())
else:
    print("No trades in first 1000 rows — trying more rows")
    df2 = next(store.to_df(count=10000)).reset_index()
    trades2 = df2[df2["action"] == "T"][["ts_event", "price", "size"]].head(5)
    print(trades2)
    print("price / 1e9 =", (trades2["price"] / 1e9).tolist())

=== Timestamp precision (raw ns) ===
[1746144901156011554, 1746144901156011554, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000, 1746144000000000000]

Last 3 digits of ts_event (000 = microsecond precision):
ts_event
0      215
373     11
465     10
229     10
641      8
      ... 
673      1
209      1
909      1
719      1
399      1
Name: count, Length: 349, dtype: int64

Last 3 digits of ts_recv (000 = microsecond precision):
ts_recv
381    26
469    26
982    25
55     24
954    23
       ..
200     1
441     1
654     1
855     1
588     1
Name: count, Length: 422, dtype: int64

=== Channel IDs ===
channel_id
15    1000
Name: count, dtype: int64

=== Flags ===
flags
128    581


In [17]:
# Check actual dtype and raw integer value before any division
print("price dtype:", df["price"].dtype)
print("price raw values (first 10 trades):")
trades_raw = df[df["action"] == "T"]["price"].head(10)
print(trades_raw.tolist())

# Also check instrument definition for price scale factor
# The scale factor (display_factor) is in the definition records
store2 = db.DBNStore.from_file(data_folder / "xeur-eobi-20250502.mbo.dbn.zst")
defs = store2.to_df(schema="definition").reset_index() if hasattr(store2, 'to_df') else None
if defs is not None:
    fdax_def = defs[defs["raw_symbol"].str.contains("FDAX", na=False)].head(3)
    print("\n=== Instrument definition (FDAX) ===")
    print(fdax_def[["raw_symbol", "min_price_increment", "display_factor", "price_ratio"]].to_string())

price dtype: float64
price raw values (first 10 trades):
[22799.0, 22798.0, 22797.0, 22799.0, 22798.0, 22800.0, 22800.0, 22800.0, 22800.0, 22800.0]

=== Instrument definition (FDAX) ===
Empty DataFrame
Columns: [raw_symbol, min_price_increment, display_factor, price_ratio]
Index: []


In [18]:
store = db.DBNStore.from_file(data_folder / "xeur-eobi-20250502.mbo.dbn.zst")

print(dir(store))

print("\n=== Available methods ===")
methods = [m for m in dir(store) if not m.startswith("_")]
print(methods)

import inspect
print("\n=== to_df signature ===")
print(inspect.signature(store.to_df))

['DBN_READ_SIZE', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_compression', '_data_source', '_instrument_map', '_metadata', '_metadata_length', '_schema_struct_map', '_transcode', 'compression', 'dataset', 'end', 'from_bytes', 'from_file', 'insert_symbology_json', 'limit', 'mappings', 'metadata', 'nbytes', 'raw', 'reader', 'replay', 'request_full_definitions', 'request_symbology', 'schema', 'start', 'stype_in', 'stype_out', 'symbology', 'symbols', 'to_csv', 'to_df', 'to_file', 'to_json', 'to_ndarray', 'to_parquet']

=== Available methods ===
['DBN_READ_SIZE', 'compression', 'dataset', 'end', 'from_bytes', 'from_file', 'insert_symbology_json', 'limit', 'mappings'

In [20]:
import databento.common.types as dbt
print(dir(dbt))

['BentoWarning', 'Callable', 'ClientRecordCallback', 'ClientStream', 'Default', 'ExceptionCallback', 'Generic', 'IO', 'MappingIntervalDict', 'PathLike', 'ReconnectCallback', 'RecordCallback', 'TypeVar', 'TypedDict', '_T', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'databento_dbn', 'dt', 'logger', 'logging', 'pd', 'validate_file_write_path', 'warnings']


In [21]:
import databento_dbn
print([x for x in dir(databento_dbn) if "rice" in x or "price" in x or "Price" in x])

# Et check la signature to_df directement sur l'enum attendu
store = db.DBNStore.from_file(data_folder / "xeur-eobi-20250502.mbo.dbn.zst")

# Test direct avec la string — la signature dit 'PriceType | str' donc str devrait marcher
print("\n=== Test price_type=fixed (string) ===")
df_fixed = next(store.to_df(price_type="fixed", pretty_ts=False, count=1000)).reset_index()
print("price dtype:", df_fixed["price"].dtype)
print("ts_recv dtype:", df_fixed["ts_recv"].dtype)
trades_fixed = df_fixed[df_fixed["action"] == "T"]["price"].head(5)
print("raw prices:", trades_fixed.tolist())

# to_parquet signature
import inspect
print("\n=== to_parquet signature ===")
print(inspect.signature(store.to_parquet))

[]

=== Test price_type=fixed (string) ===
price dtype: int64
ts_recv dtype: uint64
raw prices: [22799000000000, 22798000000000, 22797000000000, 22799000000000, 22798000000000]

=== to_parquet signature ===
(path: 'PathLike[str] | str', price_type: 'PriceType | str' = <PriceType.FLOAT: 'float'>, pretty_ts: 'bool' = True, map_symbols: 'bool' = True, schema: 'Schema | str | None' = None, mode: "Literal['w', 'x']" = 'w', parquet_schema: 'pa.Schema | None' = None, **kwargs: 'Any') -> 'None'


In [22]:
# Check to_ndarray — utile pour le split sans passer par pandas
print("=== to_ndarray signature ===")
print(inspect.signature(store.to_ndarray))

# Test to_parquet natif sur un fichier temporaire
import tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())

store.to_parquet(
    tmp / "test_fdax.parquet",
    price_type="fixed",
    pretty_ts=False,
    map_symbols=True,
)

# Read back and inspect
import pyarrow.parquet as pq
tbl = pq.read_table(tmp / "test_fdax.parquet")
print("\n=== Schema from native to_parquet ===")
print(tbl.schema)
print("\n=== Row count ===")
print(len(tbl))
print("\n=== Action distribution ===")
import pyarrow.compute as pc
actions = tbl.column("action")
print(pc.value_counts(actions))

=== to_ndarray signature ===
(schema: 'Schema | str | None' = None, count: 'int | None' = None) -> 'np.ndarray[Any, Any] | NDArrayIterator'

=== Schema from native to_parquet ===
ts_event: uint64
rtype: uint8
publisher_id: uint16
instrument_id: uint32
action: large_string
side: large_string
price: int64
size: uint32
channel_id: uint8
order_id: uint64
flags: uint8
ts_in_delta: int32
sequence: uint32
symbol: large_string
ts_recv: uint64
-- schema metadata --
pandas: '{"index_columns": ["ts_recv"], "column_indexes": [{"name": null,' + 1908

=== Row count ===
3203614

=== Action distribution ===
-- is_valid: all not null
-- child 0 type: large_string
  [
    "R",
    "T",
    "A",
    "C",
    "F",
    "M"
  ]
-- child 1 type: int64
  [
    12,
    32220,
    1568157,
    1569332,
    32189,
    1704
  ]
